In [ ]:
%pip install langchain
%pip install pinecone
%pip install pypdf
%pip install openai
%pip install tiktoken
%pip install langchain-pinecone 
%pip install langchain-community 
%pip install langchain-text-splitters

In [1]:
import langchain, langchain_openai, langchain_pinecone, pinecone, pypdf, tiktoken
print("All good")

All good


In [ ]:
import os

from langchain_community.document_loaders import PyPDFDirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_pinecone import PineconeVectorStore
from langchain_classic.chains import RetrievalQA
from langchain_core.prompts import PromptTemplate
from pinecone import Pinecone

In [3]:
!mkdir pdfs

A subdirectory or file pdfs already exists.


In [4]:
loader = PyPDFDirectoryLoader("pdfs/")

In [5]:
data =loader.load()

In [ ]:
data[0]

In [7]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=500,chunk_overlap=20)

In [8]:
text_chunks = text_splitter.split_documents(data)

In [ ]:
text_chunks

In [ ]:
print(text_chunks[0].page_content)

In [ ]:
print(text_chunks[1].page_content)

In [30]:
print(text_chunks[2].page_content)

less time to train. Our model achieves 28.4 BLEU on the WMT 2014 English-
to-German translation task, improving over the existing best results, including
ensembles, by over 2 BLEU. On the WMT 2014 English-to-French translation task,
our model establishes a new single-model state-of-the-art BLEU score of 41.0 after
training for 3.5 days on eight GPUs, a small fraction of the training costs of the
best models from the literature.
1 Introduction


In [31]:
print(len(text_chunks))

76


In [33]:
print(text_chunks[75].page_content)

[31] Yonghui Wu, Mike Schuster, Zhifeng Chen, Quoc V Le, Mohammad Norouzi, Wolfgang
Macherey, Maxim Krikun, Yuan Cao, Qin Gao, Klaus Macherey, et al. Google’s neural machine
translation system: Bridging the gap between human and machine translation. arXiv preprint
arXiv:1609.08144, 2016.
[32] Jie Zhou, Ying Cao, Xuguang Wang, Peng Li, and Wei Xu. Deep recurrent models with
fast-forward connections for neural machine translation. CoRR, abs/1606.04199, 2016.
11


In [12]:
import os
os.environ["GOOGLE_API_KEY"]="AQ.Ab8RN6IAeFx8XngV6CzE-FjfPMiiM77xREqXVLEH3Ef6KkEzNQ"
os.environ["PINECONE_API_KEY"]="pcsk_3CZeGm_7hSqRQcfnppgyMk58XY2LN1KAmNun1UgyR2TbkmJ4Lrp99zBHFwudxp1WKZwzC6"

In [13]:
from langchain_pinecone import PineconeEmbeddings


In [14]:
embedding = PineconeEmbeddings()

In [15]:
len(embedding.embed_query("How are you"))

1024

In [16]:

PINECONE_API_ENV = os.environ.get("PINECONE_API_ENV","gcp-starter")
PINECONE_API_KEY = os.environ.get("PINECONE_API_KEY","pcsk_3CZeGm_7hSqRQcfnppgyMk58XY2LN1KAmNun1UgyR2TbkmJ4Lrp99zBHFwudxp1WKZwzC6")

In [17]:
from pinecone import Pinecone

In [18]:
pc = Pinecone(api_key=PINECONE_API_KEY)

In [19]:
index_name = "test"

In [20]:
index = pc.Index(index_name)


Create Embeddings for each of the Text Chunk

In [21]:
docsearch = PineconeVectorStore.from_texts([t.page_content for t in text_chunks], embedding,index_name=index_name)

In [22]:
docsearch

In [23]:
query="Most competitive neural sequence"

In [24]:
docs = docsearch.similarity_search(query)

In [25]:
docs

[Document(id='d24370b0-2d16-4088-9e9d-d2b9df79260b', metadata={}, page_content='aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate\nself-attention and discuss its advantages over models such as [14, 15] and [8].\n3 Model Architecture\nMost competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].\nHere, the encoder maps an input sequence of symbol representations (x1,...,x n) to a sequence\nof continuous representations z = (z1,...,z n). Given z, the decoder then generates an output'),
 Document(id='d198af2f-5c87-4b68-a70c-1f4d199bd1ba', metadata={}, page_content='aligned RNNs or convolution. In the following sections, we will describe the Transformer, motivate\nself-attention and discuss its advantages over models such as [14, 15] and [8].\n3 Model Architecture\nMost competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].\nHere, the encoder maps an input sequence o

In [26]:
from langchain_openai import OpenAI
API_KEY = os.environ["GOOGLE_API_KEY"]="AQ.Ab8RN6IAeFx8XngV6CzE-FjfPMiiM77xREqXVLEH3Ef6KkEzNQ"

llm = ChatOpenAI(api_key=API_KEY,base_url="https://generativelanguage.googleapis.com/v1beta/openai/",model="gemini-3.5-flash-lite")


In [27]:
qa = RetrievalQA.from_chain_type(llm=llm,chain_type="stuff",retriever=docsearch.as_retriever())

In [29]:
qa.invoke(query)

{'query': 'Most competitive neural sequence',
 'result': 'Most competitive neural sequence transduction models have an encoder-decoder structure [5, 2, 29].'}

In [ ]:
while True:
    user_input = input(f"Input Prompt: ")
    if user_input == 'exit':
        print("Exiting")
        break
    if user_input == '':
        continue
    result = qa.invoke({'query': user_input})
    print(f"Answer: {result['result']}")

Answer: I don't know the answer to that.
Answer: Most competitive neural sequence transduction models have an encoder-decoder structure.
Answer: I don't know your name based on the provided context.
Exiting
